# 🔁 Recurrent Neural Networks (RNNs) — The Complete Deep-Dive

**Topics covered (in order):**
1. Failure of Dense networks & why we need RNNs
2. What IS an RNN — definition, intuition, unrolling
3. The three weight matrices (components)
4. Step-by-step forward-pass computation (numerical trace)
5. Architecture taxonomy (One-to-One through Many-to-Many)
6. Uni-directional vs Bi-directional RNN
7. Backpropagation Through Time (BPTT)
8. Full worked application — Sentiment Analysis
9. Issues in RNN + mathematical proofs
10. Visualisations for every concept
11. Interview Q&A


## 1. 🧱 Why Dense / MLP Networks Fail on Sequences

### 1.1 The Fixed-Size Problem
A standard fully-connected (Dense / MLP) network takes a **fixed-size vector** as input.  
- "Good" → 1 word → pad to length 10 → `[tok, 0, 0, 0, 0, 0, 0, 0, 0, 0]`  
- "The movie was absolutely breathtaking and beautiful" → 7 words → fits  
- "The film, directed by the award-winning Christopher Nolan, is …"  → 100+ words → **truncated → information lost**

You can pick a very large fixed size, but you waste memory on short sentences AND you still have an upper limit.

### 1.2 The No-Memory / No-Order Problem
Imagine a Dense network reading the word IDs `[4, 9, 2]` for "dog bites man" and `[2, 9, 4]` for "man bites dog".  
To a Dense layer, these **look the same** after a global pooling — both contain word IDs 2, 4, 9.  
**Order is destroyed** by the matrix multiply of a flat input vector.

### 1.3 The Context Problem
"I grew up in France … I speak fluent ___"  
The blank is "French" — but the clue ("France") is 8 words earlier.  
A Dense network has **no mechanism** to carry that context forward.

> **Summary table**
> | Problem | Dense Network | RNN |
> |---|---|---|
> | Variable-length input | ❌ Requires padding to fixed size | ✅ Processes any length |
> | Word order / sequence order | ❌ Treats input as a bag | ✅ Steps through in order |
> | Long-range context | ❌ No memory across positions | ✅ Hidden state carries context |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f0f1a')
for ax in axes:
    ax.set_facecolor('#0f0f1a')
    ax.axis('off')

# ── Left: Dense Network (flat, no order) ──
ax = axes[0]
ax.set_title("Dense Network\n(processes bag of words — NO order)", color='#ff6b6b', fontsize=13, pad=10)
positions_in  = [(0.1, 0.2 + i*0.15) for i in range(4)]
positions_hid = [(0.5, 0.15 + i*0.18) for i in range(4)]
positions_out = [(0.9, 0.4)]
for (x1,y1) in positions_in:
    for (x2,y2) in positions_hid:
        ax.annotate("", xy=(x2,y2), xytext=(x1,y1),
                    arrowprops=dict(arrowstyle="-", color='#555577', lw=0.7))
for (x1,y1) in positions_hid:
    for (x2,y2) in positions_out:
        ax.annotate("", xy=(x2,y2), xytext=(x1,y1),
                    arrowprops=dict(arrowstyle="-", color='#555577', lw=0.7))
words = ["[dog]","[bites]","[man]","[PAD]"]
for idx,(x,y) in enumerate(positions_in):
    ax.add_patch(plt.Circle((x,y), 0.04, color='#4ecdc4'))
    ax.text(x-0.09, y, words[idx], color='white', fontsize=9, va='center')
for (x,y) in positions_hid:
    ax.add_patch(plt.Circle((x,y), 0.04, color='#a29bfe'))
ax.add_patch(plt.Circle(positions_out[0], 0.05, color='#fd79a8'))
ax.text(0.9, 0.33, 'Output', color='white', fontsize=9, ha='center')
ax.text(0.5, 0.02, '⚠ order destroyed — "dog bites man"\n  looks same as "man bites dog"',
        color='#ff6b6b', fontsize=9, ha='center')
ax.set_xlim(0,1); ax.set_ylim(0,1)

# ── Right: RNN (sequential, ordered) ──
ax = axes[1]
ax.set_title("Recurrent Network\n(processes step-by-step — preserves order)", color='#55efc4', fontsize=13, pad=10)
words_r = ["dog","bites","man"]
xt = [0.15, 0.45, 0.75]
ht = [0.5,  0.5,  0.5]
h_y = 0.5
for i, (x, w) in enumerate(zip(xt, words_r)):
    ax.add_patch(plt.FancyBboxPatch((x-0.07, 0.2), 0.14, 0.12,
                                    boxstyle="round,pad=0.01", color='#4ecdc4'))
    ax.text(x, 0.26, f'x{i+1}\n[{w}]', color='#0f0f1a', fontsize=9, ha='center', va='center')
    ax.add_patch(plt.FancyBboxPatch((x-0.07, 0.55), 0.14, 0.12,
                                    boxstyle="round,pad=0.01", color='#a29bfe'))
    ax.text(x, 0.61, f'h{i+1}', color='white', fontsize=10, ha='center', va='center')
    ax.annotate("", xy=(x, 0.55), xytext=(x, 0.32),
                arrowprops=dict(arrowstyle="->", color='#fdcb6e', lw=1.5))
for i in range(len(xt)-1):
    ax.annotate("", xy=(xt[i+1]-0.07, 0.61), xytext=(xt[i]+0.07, 0.61),
                arrowprops=dict(arrowstyle="->", color='#fd79a8', lw=2))
    ax.text((xt[i]+xt[i+1])/2, 0.68, 'memory', color='#fd79a8', fontsize=8, ha='center')
ax.text(0.5, 0.05, '✅ order preserved — each step sees previous hidden state',
        color='#55efc4', fontsize=9, ha='center')
ax.set_xlim(0,1); ax.set_ylim(0,1)

plt.tight_layout()
plt.savefig('notes/assets/dnn_vs_rnn_viz.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print("Saved → notes/assets/dnn_vs_rnn_viz.png")


## 2. 🔄 What IS a Recurrent Neural Network?

### Definition
An **RNN** is a neural network that has a **feedback loop**: the output (hidden state) from one time step is fed back as an additional input to the next time step.

```
h_t = f( W_xh · x_t  +  W_hh · h_{t-1}  +  b_h )
y_t = W_hy · h_t  +  b_y
```

where:
- `x_t`     = input vector at time step t
- `h_t`     = hidden state at time t  (the "memory")
- `h_{t-1}` = hidden state from previous step
- `y_t`     = output at time t
- `f`       = non-linear activation (usually `tanh`)

### The Unrolling Intuition
The same network (same weights) is **applied repeatedly** — once per time step.  
We can "unroll" this loop in time to visualise it as a deep network where every layer shares weights:

```
x1 ──► [RNN cell] ──h1──► [RNN cell] ──h2──► [RNN cell] ──h3──► Output
                 ↑                   ↑                   ↑
              (same W)            (same W)            (same W)
```

Key insight: **weight sharing across time** means:
- Fewer parameters than a full sequence-length network
- BUT also the source of vanishing/exploding gradients (covered in Section 9)

### The "Reading a Book" Analogy
Think of an RNN as a **person reading a book word by word**:
- `x_t` = the word they just read
- `h_{t-1}` = their "working memory" of everything read so far
- `h_t` = updated memory after reading the current word
- `y_t` = their guess/answer at this moment

The same brain (same weights `W`) is used to process every single word.


## 3. ⚙️ Components of an RNN — The Three Weight Matrices

An RNN cell has exactly **three learnable weight matrices** and **two bias vectors**:

| Parameter | Shape | Role |
|---|---|---|
| `W_xh` | `[hidden_dim, input_dim]` | Maps **current input** → hidden state |
| `W_hh` | `[hidden_dim, hidden_dim]` | Maps **previous hidden state** → current hidden state |
| `b_h`  | `[hidden_dim]`            | Hidden state bias |
| `W_hy` | `[output_dim, hidden_dim]`| Maps **hidden state** → output |
| `b_y`  | `[output_dim]`            | Output bias |

### The Full Equations
$$h_t = \tanh(W_{xh} \cdot x_t + W_{hh} \cdot h_{t-1} + b_h)$$
$$y_t = W_{hy} \cdot h_t + b_y$$

### Why `tanh` as activation?
- Output is bounded in `[-1, 1]` — prevents values from growing unboundedly
- Zero-centered — gradients can be positive or negative  
- **Downside:** derivative `sech²(z)` is always `≤ 1` — contributes to vanishing gradients  

### Total Parameter Count
For input_dim=10, hidden_dim=32, output_dim=1:
- `W_xh` = 10 × 32 = 320
- `W_hh` = 32 × 32 = 1024  ← **dominant term**
- `W_hy` = 32 × 1  = 32
- Biases = 32 + 1  = 33
- **Total = 1,409 parameters** (regardless of sequence length!)


In [ ]:
import numpy as np

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

# ── Tiny RNN: input_dim=3, hidden_dim=4, output_dim=1 ──
input_dim, hidden_dim, output_dim = 3, 4, 1

W_xh = np.random.randn(hidden_dim, input_dim) * 0.5
W_hh = np.random.randn(hidden_dim, hidden_dim) * 0.5
b_h  = np.zeros(hidden_dim)
W_hy = np.random.randn(output_dim, hidden_dim) * 0.5
b_y  = np.zeros(output_dim)

# ── Sentence: "I love dogs" → word embeddings (3-dim for demo) ──
# In real life these come from an Embedding layer
x1 = np.array([0.8, 0.1, 0.2])   # "I"
x2 = np.array([0.3, 0.9, 0.4])   # "love"
x3 = np.array([0.1, 0.2, 0.95])  # "dogs"

sentence = [("I",    x1),
            ("love", x2),
            ("dogs", x3)]

h_prev = np.zeros(hidden_dim)      # h0 = zero vector

print("=" * 60)
print("STEP-BY-STEP RNN FORWARD PASS")
print("Sentence: 'I love dogs'  |  Many-to-One (sentiment)")
print("=" * 60)

for t, (word, x_t) in enumerate(sentence, 1):
    # ── Core equation ──
    z_t = W_xh @ x_t + W_hh @ h_prev + b_h
    h_t = np.tanh(z_t)

    print(f"\n── Step {t}: word = '{word}' ──")
    print(f"  x_{t}        = {x_t}")
    print(f"  h_{t-1} (prev) = {h_prev.round(4)}")
    print(f"  W_xh·x_{t}  = {(W_xh @ x_t).round(4)}")
    print(f"  W_hh·h_{t-1}= {(W_hh @ h_prev).round(4)}")
    print(f"  z_{t} (pre-tanh) = {z_t.round(4)}")
    print(f"  h_{t} (tanh)     = {h_t.round(4)}")

    h_prev = h_t

# ── Final prediction (many-to-one: use last hidden state) ──
y_hat = W_hy @ h_prev + b_y
prob  = 1 / (1 + np.exp(-y_hat[0]))   # sigmoid

print("\n" + "=" * 60)
print("FINAL OUTPUT (Many-to-One)")
print(f"  h_3 (final memory) = {h_prev.round(4)}")
print(f"  W_hy·h_3 + b_y     = {y_hat[0]:.4f}")
print(f"  Sentiment score (σ) = {prob:.4f}  →  {'Positive 😊' if prob > 0.5 else 'Negative 😢'}")
print("=" * 60)


## 4. 🏛️ RNN Architecture Taxonomy

Not all sequence problems have the same input→output shape. RNNs handle **five** canonical patterns:

| Type | Input | Output | Example |
|---|---|---|---|
| **One-to-One** | Single vector | Single vector | Plain MLP (no sequence) |
| **One-to-Many** | Single vector | Sequence | Image → Caption (Image Captioning) |
| **Many-to-One** | Sequence | Single vector | Sentence → Positive/Negative (Sentiment) |
| **Many-to-Many (sync)** | Sequence | Sequence (same length) | POS tagging, NER |
| **Many-to-Many (async)** | Sequence | Sequence (diff length) | Machine Translation (Encoder-Decoder) |

### Architecture Diagrams (text form)
```
One-to-One:        x ──► RNN ──► y

One-to-Many:       x ──► RNN ──► y1 ──► y2 ──► y3
                          └─h1──►┘ └─h2──►┘

Many-to-One:       x1 ──► x2 ──► x3 ──► RNN ──► y

Many-to-Many:      x1 ──► x2 ──► x3
(sync)             y1     y2     y3   (output at every step)

Many-to-Many:      x1─x2─x3 ──[encoder]──h ──[decoder]──► y1─y2─y3
(async/seq2seq)
```

### Choosing the Right Architecture
- **Sentiment / Spam / Classification** → Many-to-One  
- **Text Generation / Music** → One-to-Many  
- **NER / POS / Speech Recognition** → Many-to-Many (sync)  
- **Translation / Summarisation** → Many-to-Many (async / Seq2Seq)  


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor('#0f0f1a')
fig.suptitle("RNN Architecture Types", color='white', fontsize=15, fontweight='bold')

COLORS = {'x': '#4ecdc4', 'h': '#a29bfe', 'y': '#fd79a8', 'arrow': '#fdcb6e'}

def draw_box(ax, cx, cy, label, color, size=0.08):
    ax.add_patch(plt.FancyBboxPatch((cx-size, cy-size/2),
                                    size*2, size,
                                    boxstyle="round,pad=0.01", color=color, zorder=3))
    ax.text(cx, cy-size/4, label, color='black', fontsize=9,
            ha='center', va='center', fontweight='bold', zorder=4)

def arrow(ax, x1, y1, x2, y2):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="->", color=COLORS['arrow'], lw=1.5))

# ── Many-to-One ──
ax = axes[0]
ax.set_facecolor('#0f0f1a'); ax.axis('off')
ax.set_title("Many-to-One
(Sentiment Analysis)", color='#55efc4', fontsize=11)
xs = [0.15, 0.42, 0.68]
for i, x in enumerate(xs):
    draw_box(ax, x, 0.3, f'x{i+1}', COLORS['x'])
    draw_box(ax, x, 0.6, f'h{i+1}', COLORS['h'])
    arrow(ax, x, 0.34, x, 0.56)
    if i < len(xs)-1:
        arrow(ax, xs[i]+0.08, 0.6, xs[i+1]-0.08, 0.6)
draw_box(ax, 0.88, 0.6, 'y', COLORS['y'])
arrow(ax, xs[-1]+0.08, 0.6, 0.8, 0.6)
ax.text(0.5, 0.1, '"I love this movie" → Positive', color='#aaa', fontsize=9, ha='center')
ax.set_xlim(0,1); ax.set_ylim(0,1)

# ── Many-to-Many Sync ──
ax = axes[1]
ax.set_facecolor('#0f0f1a'); ax.axis('off')
ax.set_title("Many-to-Many (sync)
(Named Entity Recognition)", color='#55efc4', fontsize=11)
xs = [0.2, 0.5, 0.8]
for i, x in enumerate(xs):
    draw_box(ax, x, 0.25, f'x{i+1}', COLORS['x'])
    draw_box(ax, x, 0.55, f'h{i+1}', COLORS['h'])
    draw_box(ax, x, 0.82, f'y{i+1}', COLORS['y'])
    arrow(ax, x, 0.29, x, 0.51)
    arrow(ax, x, 0.59, x, 0.78)
    if i < len(xs)-1:
        arrow(ax, xs[i]+0.08, 0.55, xs[i+1]-0.08, 0.55)
ax.text(0.5, 0.07, '"London is great" → [LOC, O, O]', color='#aaa', fontsize=9, ha='center')
ax.set_xlim(0,1); ax.set_ylim(0,1)

# ── One-to-Many ──
ax = axes[2]
ax.set_facecolor('#0f0f1a'); ax.axis('off')
ax.set_title("One-to-Many
(Image Captioning)", color='#55efc4', fontsize=11)
draw_box(ax, 0.15, 0.55, 'img', COLORS['x'], size=0.1)
hs = [0.35, 0.57, 0.78]
ys_pos = [0.35, 0.57, 0.78]
for i, hx in enumerate(hs):
    draw_box(ax, hx, 0.55, f'h{i+1}', COLORS['h'])
    draw_box(ax, hx, 0.82, f'y{i+1}', COLORS['y'])
    arrow(ax, hx, 0.59, hx, 0.78)
    if i == 0:
        arrow(ax, 0.25, 0.55, hs[0]-0.08, 0.55)
    if i < len(hs)-1:
        arrow(ax, hs[i]+0.08, 0.55, hs[i+1]-0.08, 0.55)
ax.text(0.6, 0.07, 'Image → "A dog runs fast"', color='#aaa', fontsize=9, ha='center')
ax.set_xlim(0,1); ax.set_ylim(0,1)

plt.tight_layout()
plt.savefig('notes/assets/rnn_architectures.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print("Saved → notes/assets/rnn_architectures.png")


## 5. ↔️ Uni-directional vs Bi-directional RNN

### Uni-directional (Standard)
- Reads the sequence **left → right** only
- At each step `t`, the hidden state `h_t` only has access to words `x_1 … x_t`
- **Cannot see the future**

**Best for:**
- Real-time / streaming data (speech-to-text, live translation)
- Time-series forecasting (you cannot use future stock prices to predict today's)
- Text generation (auto-regressive — next word depends only on past)

### Bi-directional RNN
Runs **two** RNN layers simultaneously:
- Forward RNN: reads `x_1 → x_2 → … → x_T`  (left to right)
- Backward RNN: reads `x_T → x_{T-1} → … → x_1`  (right to left)

At each position `t`, the final representation is:
$$H_t = [\overrightarrow{h}_t \; ; \; \overleftarrow{h}_t]$$
(concatenation of both directions — doubles the hidden dimension)

**Best for:**
- Tasks where **full sentence context** is available at inference time
- Sentiment Analysis, Named Entity Recognition, POS tagging
- BERT uses Bidirectional Transformers — same philosophy

**Cannot use for:**
- Real-time inference (you need the full sequence before processing)
- Language modelling / text generation (would cheat by seeing future words)

### Example: Why future context matters
```
"The crane [flew / lifted] …"
```
Without future context, "crane" is ambiguous (bird or machine?).  
A bidirectional RNN resolves this by reading both directions before classifying.


In [ ]:
import torch
import torch.nn as nn

class BiRNNSentiment(nn.Module):
    """Bidirectional Many-to-One RNN for sentiment classification."""
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,   # ← key flag
            dropout=0.3,
        )
        # *2 because we concatenate forward + backward final states
        self.classifier = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        # x: (batch, seq_len)
        emb = self.embedding(x)          # (batch, seq_len, embed_dim)
        out, hidden = self.rnn(emb)
        # hidden shape: (num_layers * 2, batch, hidden_dim)
        # Take the final layer's forward and backward states
        fwd = hidden[-2, :, :]           # last forward  layer
        bwd = hidden[-1, :, :]           # last backward layer
        context = torch.cat([fwd, bwd], dim=1)  # (batch, hidden*2)
        logit = self.classifier(context)
        return torch.sigmoid(logit)

model = BiRNNSentiment(vocab_size=5000, embed_dim=64, hidden_dim=128)
dummy_input = torch.randint(1, 5000, (8, 20))   # batch=8, seq_len=20
output = model(dummy_input)

print("Model Architecture:")
print(model)
print(f"\nInput  shape : {dummy_input.shape}  (batch=8, seq_len=20)")
print(f"Output shape : {output.shape}            (batch=8, 1 probability each)")
total = sum(p.numel() for p in model.parameters())
print(f"Total params : {total:,}")


## 6. 🔙 Backpropagation Through Time (BPTT)

BPTT is the algorithm used to train RNNs. It's just standard backprop applied to the **unrolled** computation graph.

### Step-by-step process:
**① Forward pass** — compute all hidden states and outputs left to right:
$$h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b_h), \quad t = 1 \ldots T$$
$$y_t = W_{hy} h_t + b_y$$
$$L = \sum_{t} \mathcal{L}(y_t, \hat{y}_t)$$

**② Backward pass** — propagate the loss gradient backwards through time:

Start from the final loss $L$ and compute $\frac{\partial L}{\partial W_{hh}}$ using the chain rule:
$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \frac{\partial L_t}{\partial W_{hh}}$$

Each term involves the chain:
$$\frac{\partial L_t}{\partial W_{hh}} = \frac{\partial L_t}{\partial y_t} \cdot \frac{\partial y_t}{\partial h_t} \cdot \frac{\partial h_t}{\partial W_{hh}}$$

The **critical chain** for gradient flow from step $T$ all the way back to step $k$:
$$\frac{\partial h_T}{\partial h_k} = \prod_{i=k+1}^{T} \frac{\partial h_i}{\partial h_{i-1}}$$

Since $h_i = \tanh(W_{hh} h_{i-1} + \ldots)$:
$$\frac{\partial h_i}{\partial h_{i-1}} = \text{diag}(\tanh'(z_i)) \cdot W_{hh}$$

So:
$$\frac{\partial h_T}{\partial h_k} = \prod_{i=k+1}^{T} \text{diag}(\tanh'(z_i)) \cdot W_{hh}$$

This is a **product of** ($T - k$) **matrices** — the source of all RNN training difficulty (covered next).

### Truncated BPTT
In practice, for very long sequences we use **Truncated BPTT**:
- Process the sequence in chunks of `k` steps  
- Backpropagate only `k` steps back  
- Reset gradients between chunks  
- **Trade-off:** Less accurate gradients, but computationally feasible


## 7. 🚨 Issues in RNN — With Mathematical Proofs

### Issue 1: Vanishing Gradient

#### The Proof (Scalar simplified case)
Assume linear activation for clarity. The gradient from step $T$ to step $k$:
$$\frac{\partial h_T}{\partial h_k} = W_{hh}^{T-k}$$

**Case 1 — Vanishing ($|W_{hh}| < 1$):**
$$W_{hh} = 0.9, \quad T - k = 50 \implies (0.9)^{50} \approx 0.005$$

The gradient arriving at step $k$ is **200× smaller** than at step $T$.  
Weights at early time steps **never get updated** — the network forgets long-range context.

**Case 2 — With tanh (reality):**
$\tanh'(z) = 1 - \tanh^2(z) \in [0, 1]$ always.  
Each step multiplies by both $W_{hh}$ AND $\tanh'(z_i) \leq 1$.  
This makes vanishing **even faster** than the scalar analysis suggests.

**Why does it matter?**
- The model learns short-range patterns (last 3-5 words) but not long-range ones (50+ words)
- "The animal didn't cross the street because **it** was too tired" — the word "it" refers to "animal" many words earlier → vanilla RNN fails here

#### The Jacobian (Full Vector Proof)
For vector hidden states:
$$\left\| \frac{\partial h_T}{\partial h_k} \right\| \leq \prod_{i=k+1}^{T} \left\| \text{diag}(\tanh'(z_i)) \right\| \cdot \left\| W_{hh} \right\|$$
$$\leq (\lambda_{max}(W_{hh}))^{T-k}$$

where $\lambda_{max}$ is the largest eigenvalue. If $\lambda_{max} < 1$ → gradient **vanishes exponentially**.

---

### Issue 2: Exploding Gradient

**When $|W_{hh}| > 1$:**
$$W_{hh} = 1.1, \quad T - k = 100 \implies (1.1)^{100} \approx 13,780$$

Gradients become astronomically large → weight updates destroy the model.

**Solution: Gradient Clipping**
$$\text{if } \|g\| > \theta \implies g \leftarrow \frac{\theta}{\|g\|} \cdot g$$

Hard-cap the gradient norm to a maximum value (commonly $\theta = 5.0$).

---

### Issue 3: Short-Term Memory
At every step, $h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t)$ **overwrites** the hidden state.  
New information in $x_t$ can completely overwrite what was in $h_{t-1}$.  
There is **no gating mechanism** to say "keep this fact, discard that one."  
→ Fixed by LSTM (notebook 10) and GRU (notebook 11).

---

### Issue 4: Sequential / Non-Parallelisable Computation
$h_t$ depends on $h_{t-1}$ — you **cannot** compute step $t$ without finishing step $t-1$.  
This blocks GPU parallelism across the time dimension.  
Transformers (attention-based) solve this by removing recurrence entirely.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#0f0f1a')
fig.suptitle("RNN Training Issues — Visualised", color='white', fontsize=14)

# ── Plot 1: Vanishing vs Exploding gradient ──
ax = axes[0]
ax.set_facecolor('#13131f')
steps = np.arange(1, 51)

for w, lbl, col in [(0.7, 'W=0.7 (fast vanish)', '#ff6b6b'),
                     (0.95,'W=0.95 (slow vanish)','#fdcb6e'),
                     (1.0, 'W=1.0 (stable)',      '#55efc4'),
                     (1.05,'W=1.05 (explode)',     '#a29bfe')]:
    grads = w ** steps
    ax.plot(steps, np.clip(grads, 0, 20), label=lbl, color=col, lw=2)

ax.set_xlabel("Steps back in time", color='#aaa')
ax.set_ylabel("Gradient magnitude", color='#aaa')
ax.set_title("Vanishing vs Exploding Gradients
|W_hh|^t", color='white')
ax.legend(fontsize=8)
ax.tick_params(colors='#aaa')
ax.set_ylim(0, 12)
for sp in ax.spines.values(): sp.set_color('#333')

# ── Plot 2: tanh derivative shrinkage ──
ax = axes[1]
ax.set_facecolor('#13131f')
z = np.linspace(-4, 4, 300)
tanh_deriv = 1 - np.tanh(z)**2
ax.plot(z, np.tanh(z), color='#4ecdc4', lw=2, label='tanh(z)')
ax.plot(z, tanh_deriv,  color='#fd79a8', lw=2, label="tanh'(z) = 1-tanh²(z)")
ax.axhline(1, color='#fdcb6e', lw=1, ls='--', label='max=1 at z=0')
ax.fill_between(z, tanh_deriv, 0, alpha=0.15, color='#fd79a8')
ax.set_title("tanh Derivative Always ≤ 1
(Every step shrinks gradient)", color='white')
ax.set_xlabel("z (pre-activation value)", color='#aaa')
ax.legend(fontsize=9)
ax.tick_params(colors='#aaa')
ax.set_ylim(-1.2, 1.2)
for sp in ax.spines.values(): sp.set_color('#333')

# ── Plot 3: Effect on "memory" at different distances ──
ax = axes[2]
ax.set_facecolor('#13131f')
words = ['The','cat','that','ate','the','large','rat','is','sick']
distances = np.arange(len(words), 0, -1)   # distance from prediction
# gradient reaching each position (W=0.85, tanh shrink≈0.5 per step)
effective_grad = (0.85 * 0.7) ** distances

colors_bar = ['#ff6b6b' if g < 0.01 else '#fdcb6e' if g < 0.1 else '#55efc4'
              for g in effective_grad]
bars = ax.barh(words[::-1], effective_grad[::-1], color=colors_bar[::-1])
ax.axvline(0.01, color='white', ls='--', lw=1, label='≈0.01 threshold (barely learning)')
ax.set_title('Effective Gradient per Word
(Many-to-One, last step prediction)', color='white')
ax.set_xlabel('Gradient magnitude reaching this word', color='#aaa')
ax.legend(fontsize=8)
ax.tick_params(colors='#aaa')
for sp in ax.spines.values(): sp.set_color('#333')

plt.tight_layout()
plt.savefig('notes/assets/rnn_issues.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print("Saved → notes/assets/rnn_issues.png")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0f0f1a')

# ── Simulate training WITHOUT clipping ──
np.random.seed(7)
def simulate_rnn_training(clip=False, threshold=5.0, steps=60):
    W = 1.05   # slightly > 1 → exploding regime
    losses = []
    grads  = []
    grad   = 1.0
    loss   = 5.0
    for _ in range(steps):
        grad *= W + np.random.normal(0, 0.05)
        grads.append(abs(grad))
        if clip and abs(grad) > threshold:
            grad = threshold * np.sign(grad)
        loss -= 0.03 * grad / (abs(grad) + 1e-6) * np.random.uniform(0.5, 1.5)
        losses.append(loss)
    return losses, grads

loss_no_clip, grad_no_clip = simulate_rnn_training(clip=False)
loss_clipped, grad_clipped = simulate_rnn_training(clip=True, threshold=5.0)

for ax, title, (l1, l2, g1, g2) in zip(axes,
    ["Without Gradient Clipping", "With Gradient Clipping (threshold=5)"],
    [([loss_no_clip, loss_clipped, grad_no_clip, grad_no_clip]),
     ([loss_clipped, loss_clipped, grad_clipped, grad_clipped])]):
    ax.set_facecolor('#13131f')

axes[0].set_facecolor('#13131f')
axes[0].plot(loss_no_clip, color='#ff6b6b', lw=2, label='Loss (explodes)')
axes[0].plot(grad_no_clip, color='#fdcb6e', lw=1.5, ls='--', label='|Gradient|')
axes[0].set_title('Without Gradient Clipping', color='white')
axes[0].set_xlabel('Training step', color='#aaa')
axes[0].legend(fontsize=9); axes[0].tick_params(colors='#aaa')
axes[0].set_ylim(-5, 30)
for sp in axes[0].spines.values(): sp.set_color('#333')

axes[1].set_facecolor('#13131f')
axes[1].plot(loss_clipped, color='#55efc4', lw=2, label='Loss (stable)')
axes[1].plot(grad_clipped, color='#4ecdc4', lw=1.5, ls='--', label='|Gradient| (clipped)')
axes[1].axhline(5.0, color='#fd79a8', ls='--', lw=1.5, label='Clip threshold = 5')
axes[1].set_title('With Gradient Clipping', color='white')
axes[1].set_xlabel('Training step', color='#aaa')
axes[1].legend(fontsize=9); axes[1].tick_params(colors='#aaa')
axes[1].set_ylim(-5, 30)
for sp in axes[1].spines.values(): sp.set_color('#333')

plt.suptitle("Exploding Gradient — Before and After Clipping", color='white', fontsize=13)
plt.tight_layout()
plt.savefig('notes/assets/gradient_clipping.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()


## 8. 🛠️ End-to-End Application: Sentiment Analysis (Many-to-One)

We'll build a complete sentiment classifier:  
- Tokenise sentences  
- Embed words  
- Pass through Bidirectional RNN  
- Train with binary cross-entropy  
- Plot loss curve  


In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)

# ── Tiny synthetic dataset ──
POS = ["I love this", "great movie wonderful", "amazing film loved it",
       "fantastic brilliant superb", "excellent fun enjoyed"]
NEG = ["terrible awful film", "hated it boring bad", "worst movie ever seen",
       "dreadful horrible disgusting", "disappointed sad waste"]

# Build vocabulary
all_words = " ".join(POS + NEG).split()
vocab = {w: i+1 for i,w in enumerate(set(all_words))}
vocab['<PAD>'] = 0

def encode(sentence, max_len=6):
    tokens = [vocab.get(w, 0) for w in sentence.split()]
    tokens = tokens[:max_len] + [0]*(max_len - len(tokens))
    return tokens

max_len = 6
X = torch.tensor([encode(s, max_len) for s in POS + NEG], dtype=torch.long)
y = torch.tensor([1]*5 + [0]*5, dtype=torch.float).unsqueeze(1)

# ── Model ──
class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=32):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True,
                          bidirectional=True, num_layers=1)
        self.fc  = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        e = self.emb(x)
        _, h = self.rnn(e)        # h: (2, batch, hidden)
        ctx = torch.cat([h[0], h[1]], dim=1)
        return torch.sigmoid(self.fc(ctx))

model = SentimentRNN(vocab_size=len(vocab)+1)
optim = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.BCELoss()

# ── Training loop ──
losses = []
for epoch in range(200):
    model.train()
    pred  = model(X)
    loss  = loss_fn(pred, y)
    optim.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
    optim.step()
    losses.append(loss.item())

# ── Final accuracy ──
model.eval()
with torch.no_grad():
    preds = model(X).squeeze()
    acc = ((preds > 0.5).float() == y.squeeze()).float().mean()

# ── Plot training curve ──
fig, ax = plt.subplots(figsize=(9, 4))
fig.patch.set_facecolor('#0f0f1a')
ax.set_facecolor('#13131f')
ax.plot(losses, color='#4ecdc4', lw=2, label='Training Loss (BCE)')
ax.fill_between(range(len(losses)), losses, alpha=0.15, color='#4ecdc4')
ax.set_xlabel('Epoch', color='#aaa')
ax.set_ylabel('Binary Cross-Entropy Loss', color='#aaa')
ax.set_title(f'Sentiment RNN — Training Curve  |  Final Acc: {acc:.0%}',
             color='white', fontsize=12)
ax.legend(fontsize=10)
ax.tick_params(colors='#aaa')
for sp in ax.spines.values(): sp.set_color('#333')
plt.tight_layout()
plt.savefig('notes/assets/rnn_training_curve.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

# ── Predictions ──
print("\n── Final Predictions ──")
all_sentences = POS + NEG
with torch.no_grad():
    probs = model(X).squeeze().tolist()
for sentence, prob, label in zip(all_sentences, probs, [1]*5+[0]*5):
    pred_label = "😊 POS" if prob > 0.5 else "😢 NEG"
    correct = "✅" if (prob>0.5)==label else "❌"
    print(f"{correct} {pred_label} ({prob:.2f}) | '{sentence}'")
print(f"\nOverall Accuracy: {acc:.0%}")


## 9. 📋 RNN Issues — Complete Reference Table

| Issue | Root Cause | Mathematical Proof | Solution |
|---|---|---|---|
| **Vanishing Gradient** | $(W_{hh} \cdot \tanh')^{T-k} \to 0$ when eigenvalues $<1$ | $\|\partial h_T / \partial h_k\| \leq \lambda_{max}^{T-k}$ | LSTM gates, GRU, ResNet-style skip connections |
| **Exploding Gradient** | $(W_{hh})^{T-k} \to \infty$ when eigenvalues $>1$ | Same product, $\lambda > 1$ | Gradient clipping ($\|g\| \leq \theta$) |
| **Short-Term Memory** | $h_t$ overwrites $h_{t-1}$ with no gating | No selective forget/keep mechanism in `tanh` | LSTM (Forget gate), GRU (Update gate) |
| **Sequential Bottleneck** | $h_t$ depends on $h_{t-1}$ → no time-dim parallelism | Dependency chain blocks GPU parallelism | Transformers (self-attention removes recurrence) |
| **Padding Artefacts** | RNN processes `<PAD>` tokens as real data | Hidden state gets "polluted" after EOS | `pack_padded_sequence` + `pad_packed_sequence` |

### Edge Cases to Know for Interviews

**1. Zero-length sequences / fully padded inputs**  
Always check input length before passing to RNN. Use `pack_padded_sequence`.

**2. Exploding at init (fresh model)**  
At random init, $W_{hh}$ eigenvalues can be $>1$. Orthogonal initialisation keeps $|\lambda| = 1$ exactly.

**3. Gradient vanishing vs weight decay confusion**  
Weight decay (L2 regularisation) *intentionally* shrinks weights. This can **worsen** vanishing gradients in RNNs. Use with care.


## 10. 🎯 Senior-Level Interview Q&A

**Q1: Explain vanishing gradients without equations.**  
> "Imagine photocopying a document 100 times. Each copy is slightly blurrier. By the 100th copy, the text is unreadable. In an RNN, each time step 'photocopies' the gradient signal through the same weight matrix. If the weights are $<1$, the signal fades to zero and early steps stop learning."

---

**Q2: Why can't we just use a very large learning rate to compensate for vanishing gradients?**  
> A large LR amplifies the gradient update across *all* layers. Steps that don't suffer from vanishing (recent steps) will get overfit and destabilised, while early steps still receive near-zero gradients. The problem is asymmetric — LR can't fix it.

---

**Q3: What is the difference between BPTT and truncated BPTT? When do you use each?**  
> Full BPTT propagates gradients back across the entire sequence — accurate but expensive and prone to vanishing over very long sequences. Truncated BPTT splits the sequence into windows and backprops only a fixed number of steps back. Used for long documents or very long time-series where full BPTT is intractable.

---

**Q4: Can a unidirectional RNN be used for machine translation? Why or why not?**  
> Technically yes (seq2seq uses two unidirectional RNNs — encoder forward + decoder forward), but a single unidirectional RNN cannot do good translation alone. The encoder reads the full source sentence before the decoder starts — this gives the encoder full context. The *decoder* must still be unidirectional (left-to-right) because it generates each word before the next.

---

**Q5: What does `pack_padded_sequence` do and why is it important?**  
> It tells PyTorch to skip padded zeros during the RNN forward pass. Without it, the RNN modifies the hidden state for every `<PAD>` token, which "pollutes" the final hidden state that gets used for classification. With packing, the RNN effectively stops at the true end of each sequence.

---

**Q6: Why do we initialise the hidden state `h_0` to zeros?**  
> The hidden state represents "memory of context seen so far." At the beginning of a new sequence, there is no prior context, so a zero vector is the correct neutral starting point. Some tasks benefit from learned initial states (trained as a parameter), but zeros are the safe default.
